# RAG Finpass: jalankan pelan-pelan

Jalankan **satu sel, tunggu selesai, baru sel berikutnya**. Tidak ada proses latar belakang: semua berjalan di kernel ini.
Kernel: pilih `ai/.venv`. Tahap 0 sampai 2 ringan (tanpa model). Model baru dimuat di tahap 3.
Ingest bisa dihentikan (Interrupt) dan dilanjutkan: file dengan etag yang sama dilewati.

| Tahap | Isi | Berat |
|---|---|---|
| 0 | Setup dan cek koneksi | ringan |
| 1 | Daftar file di R2 | ringan |
| 2 | Coba satu PDF: ekstrak dan chunk | ringan |
| 3 | Muat model embedding | unduh/muat sekali |
| 4 | Ingest ojk-regulasi | sedang |
| 5 | Ingest ojk-edukasi | sedang |
| 6 | Ingest idx-dokumen (per tahun/folder) | berat |
| 7 | Status database | ringan |
| 8 | Uji retrieval (muat reranker) | sedang |
| 9 | Uji chat lengkap dengan LLM | ringan |

## Tahap 0. Setup dan cek koneksi

In [12]:
import os, sys, logging
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
os.chdir(root)                      # Settings membaca .env dari folder ai/
sys.path.insert(0, str(root))
logging.basicConfig(level="INFO", format="%(asctime)s %(levelname)s %(name)s %(message)s")
for noisy in ("httpx", "httpcore", "urllib3", "botocore", "sentence_transformers", "pdfminer"):
    logging.getLogger(noisy).setLevel("WARNING")

from src.core.config import settings
from src.features.rag import store

print("R2 bucket :", settings.r2_bucket, "| endpoint set:", bool(settings.r2_endpoint), "| key set:", bool(settings.r2_access_key_id))
print("Database  :", "set" if settings.database_url else "KOSONG (isi DATABASE_URL di ai/.env)")
print("Embed     :", settings.embed_model, "| Rerank:", settings.rerank_model)
print("LLM       :", settings.openrouter_model, "| key set:", bool(settings.openrouter_api_key))

conn = store.connect()
print("pgvector  :", conn.execute("select extversion from pg_extension where extname='vector'").fetchone())
conn.close()

R2 bucket : setagama-core | endpoint set: True | key set: True
Database  : set
Embed     : BAAI/bge-m3 | Rerank: BAAI/bge-reranker-v2-m3
LLM       : deepseek/deepseek-v4-flash | key set: True
pgvector  : ('0.8.7',)


## Tahap 1. Daftar file di R2 (tidak mengunduh apa pun)

In [13]:
import collections
from src.features.rag import r2

stats = collections.defaultdict(lambda: [0, 0])
for f in r2.list_pdfs(""):
    top = "/".join(f.key.split("/")[: 2 if f.key.startswith("idx-dokumen/") else 1])
    stats[top][0] += 1
    stats[top][1] += f.size
for k, (n, size) in sorted(stats.items()):
    print(f"{k:45s} {n:5d} file  {size/1e6:8.1f} MB")

idx-dokumen/2024                                352 file     941.7 MB
idx-dokumen/2025                                341 file     837.8 MB
idx-dokumen/2026                                669 file    1281.9 MB
ojk-edukasi                                     364 file    1202.7 MB
ojk-regulasi                                    109 file      27.4 MB


## Tahap 2. Coba satu PDF: ekstrak dan chunk
Ganti `KEY` untuk mencoba dokumen lain. Belum ada embedding dan belum menulis ke database.

In [14]:
import tempfile
from src.features.rag import chunking, extract, meta

KEY = "ojk-regulasi/Abstrak_PADK_1_Tahun_2026_Penyelenggaraan_Teknologi_Informasi_oleh_Bank_Umum.pdf"

m = meta.parse_key(KEY)
print(m)
with tempfile.TemporaryDirectory() as d:
    path = f"{d}/x.pdf"
    r2.download(KEY, path)
    pages, scanned = extract.extract_pages(path)
print("halaman:", len(pages), "| scan:", scanned)
chunks = chunking.chunk_pages(pages, m.source_type, m.title)
print("chunk  :", len(chunks))
for c in chunks[:3]:
    print(f"--- chunk {c.ordinal} hal.{c.page_start}-{c.page_end} tabel={c.has_table} section={c.section}")
    print(c.content[:400])

DocMeta(source_type='ojk_regulasi', title='Abstrak PADK 1 Tahun 2026 Penyelenggaraan Teknologi Informasi oleh Bank Umum', ticker=None, fiscal_year=2026, language='id')
halaman: 2 | scan: False
chunk  : 2
--- chunk 0 hal.1-2 tabel=False section=PADK OJK NO. 1, 3 HLM.
Abstrak PADK 1 Tahun 2026 Penyelenggaraan Teknologi Informasi oleh Bank Umum
PADK OJK NO. 1, 3 HLM.
PERBANKAN - PENYELENGGARAAN TEKNOLOGI INFORMASI – BANK UMUM
2026
PADK OJK NO. 1, 3 HLM.
PERATURAN ANGGOTA DEWAN KOMISIONER OTORITAS JASA KEUANGAN TENTANG
PENYELENGGARAAN TEKNOLOGI INFORMASI OLEH BANK UMUM
ABSTRAK : - Teknologi Informasi (TI) memiliki peranan penting dalam mendukung
operasional dan b
--- chunk 1 hal.1-2 tabel=False section=PADK OJK NO. 1, 3 HLM.
Abstrak PADK 1 Tahun 2026 Penyelenggaraan Teknologi Informasi oleh Bank Umum
PADK OJK NO. 1, 3 HLM.
- Format laporan dan notifikasi dalam penyelenggaraan TI mengacu pada
Lampiran III yang merupakan bagian tidak terpisahkan dari PADK OJK ini.
- Format permohonan izin da

## Tahap 3. Muat model embedding
Pertama kali mengunduh BGE-M3 (sekitar 2 GB) ke `~/.cache/huggingface`. Sel ini hanya memuat dan menguji satu kalimat.

In [15]:
import time
from src.features.rag import embedder

t = time.time()
v = embedder.embed(["uji embedding"])
print("dimensi:", v.shape, "| waktu:", round(time.time() - t, 1), "detik")

dimensi: (1, 1024) | waktu: 12.9 detik


## Tahap 4. Ingest ojk-regulasi
`LIMIT = 5` dulu untuk mencoba. Setelah aman, ubah ke `None` untuk semuanya (109 file kecil).
`workers=1` agar berjalan berurutan dan ringan.

In [16]:
from src.features.rag import ingest

PREFIX, LIMIT = "ojk-regulasi/", 5
ingest.run(PREFIX, LIMIT, force=False, workers=1)

2026-10-17 11:19:53,426 INFO rag.ingest 0 documents to ingest under 'ojk-regulasi/'


## Tahap 5. Ingest ojk-edukasi
364 file, median 2,7 MB. Mulai dengan `LIMIT = 10`.

In [17]:
PREFIX, LIMIT = "ojk-edukasi/", 10
ingest.run(PREFIX, LIMIT, force=False, workers=1)

2026-10-17 11:19:54,391 INFO rag.ingest 10 documents to ingest under 'ojk-edukasi/'
2026-10-17 11:20:25,797 INFO rag.ingest 10/10 docs, 120 chunks, 3.8 chunks/s, 0 skipped


## Tahap 6. Ingest idx-dokumen (laporan keuangan)
Terberat: 1.362 file, 3 GB. Jalankan satu folder per sel, dengan `LIMIT` kecil dulu.
Folder: `idx-dokumen/<2024|2025|2026>/<Financial_Statement|Laporan_Keuangan>/`.

In [18]:
PREFIX, LIMIT = "idx-dokumen/2024/Financial_Statement/", 5
ingest.run(PREFIX, LIMIT, force=False, workers=1)

2026-10-17 11:20:26,679 INFO rag.ingest 5 documents to ingest under 'idx-dokumen/2024/Financial_Statement/'
2026-10-17 11:21:44,198 INFO rag.ingest 5/5 docs, 534 chunks, 6.9 chunks/s, 0 skipped


## Tahap 7. Status database

In [19]:
conn = store.connect()
print("Dokumen per jenis (total, scan, halaman):")
for r in conn.execute("select source_type, count(*), sum(case when scanned then 1 else 0 end), coalesce(sum(pages),0) from rag_documents group by 1 order by 1"):
    print("  ", r)
print("Total chunk :", conn.execute("select count(*) from rag_chunks").fetchone()[0])
print("Ukuran tabel:", conn.execute("select pg_size_pretty(pg_total_relation_size('rag_chunks'))").fetchone()[0])
conn.close()

Dokumen per jenis (total, scan, halaman):
   ('idx_financial_statement', 10, 0, 946)
   ('ojk_edukasi', 63, 5, 2314)
   ('ojk_regulasi', 109, 0, 1192)
Total chunk : 3825
Ukuran tabel: 67 MB


## Tahap 8. Uji retrieval
Pertama kali memuat reranker BGE (sekitar 2 GB). Ubah `QUESTIONS`. Tampil: skor rerank, judul, halaman, cuplikan.

In [20]:
import time
from src.features.rag import retrieval

QUESTIONS = [
    "Apa kewajiban bank dalam penerapan manajemen risiko teknologi informasi?",
    "Apa itu diversifikasi investasi?",
]
conn = store.connect()
for q in QUESTIONS:
    t = time.time()
    passages, kind, tickers = retrieval.search(conn, q, q, "id", [])
    print(f"\nQ: {q}\n   niat={kind} ticker={tickers} waktu={time.time()-t:.1f}s")
    for p in passages[:4]:
        print(f"   {p.score:.2f}  {p.title[:60]}  hal.{p.page}: {' '.join(p.content.split()[8:24])}")
conn.close()


Q: Apa kewajiban bank dalam penerapan manajemen risiko teknologi informasi?
   niat=edukasi ticker=[] waktu=15.1s
   0.73  PADK 1 Tahun 2026 Penyelenggaraan Teknologi Informasi oleh B  hal.36: Bank Umum PENERAPAN MANAJEMEN RISIKO PENYELENGGARAAN TI BANK - 33 - BAB IV PENERAPAN MANAJEMEN RISIKO
   0.73  PADK 1 Tahun 2026 Penyelenggaraan Teknologi Informasi oleh B  hal.38: Bank Umum PENERAPAN MANAJEMEN RISIKO PENYELENGGARAAN TI BANK c. Hasil evaluasi dan tindak lanjut sebagaimana dimaksud
   0.73  PADK 1 Tahun 2026 Penyelenggaraan Teknologi Informasi oleh B  hal.36: Bank Umum PENERAPAN MANAJEMEN RISIKO PENYELENGGARAAN TI BANK Untuk dapat memastikan hal di atas, Bank dapat
   0.73  PADK 1 Tahun 2026 Penyelenggaraan Teknologi Informasi oleh B  hal.40: Bank Umum PENERAPAN MANAJEMEN RISIKO PENYELENGGARAAN TI BANK - 37 - 1. Pengamanan Informasi Pengamanan informasi

Q: Apa itu diversifikasi investasi?
   niat=edukasi ticker=[] waktu=6.8s
   0.65  PADK 41-PADK05-2025 Penerapan Manajemen Risik

## Tahap 9. Uji chat lengkap dengan LLM
Memakai OpenRouter (`OPENROUTER_MODEL`). Tanpa konteks user. Untuk menguji konteks, isi `ctx`.

In [21]:
import asyncio
from src.features.rag import service
from src.features.rag.schemas import ChatRequest, HoldingCtx, ProfileCtx, UserContext
from src.infrastructure.openrouter import OpenRouterClient

ctx = UserContext(
    profile=ProfileCtx(riskProfile="Moderate", ladderLevel=3, sharia="no", horizonYears=4),
    holdings=[HoldingCtx(symbol="BBCA.JK", name="Bank Central Asia", assetClass="equity_large_cap", weightPct=40)],
)
req = ChatRequest(message="Apa itu diversifikasi dan kenapa penting?", lang="id", userContext=ctx)

llm = OpenRouterClient() if settings.openrouter_api_key else None
out = await service.chat(req, llm)          # notebook mendukung await langsung
print("sumber:", out.source, "| grounded:", out.grounded, "| intent:", out.trace.intent)
print(out.answer)
for c in out.citations:
    print(f"[{c.n}] {c.title}, hal. {c.page} ({c.sourceType})")

2026-10-17 11:22:15,359 INFO httpx2 HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-17 11:22:16,260 INFO rag.service guardrail rejected answer: missing citation
2026-10-17 11:22:17,352 INFO httpx2 HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-17 11:22:17,855 INFO rag.service guardrail rejected answer: missing citation


sumber: fallback | grounded: True | intent: edukasi
Saya belum bisa menyusun jawaban yang lolos pemeriksaan. Berikut kutipan paling relevan dari sumber:
[1] 2015 Djap tet Fa How Financial Literacy Program From NBFI Affect Financial Inclusion, hal. 13: 3. CHALLENGE AND OPPORTUNITY TO INCREASE FINANCIAL LITERACY CHALLENGE OPPORTUNITY • Geographical & wide coverage: urban, sub-urban, • Simplification of business process, appropriate & and very remote areas. efficient delivery to customers. • Access to diversified financial produc
[2] Majalah-OJK edisi Desember 2014 th 2, hal. 33: Keberadaan perempuan amat penting. Dalam pengaturan keuangan menyampaikan perbedaan kebutuhan Bank Group) Australia menceritakan Para pembicara menyetujui keluarga, lebih banyak perempuan yang menjadi manajernya. antara UMKM dan pengusaha besar. bahwa Commonwealth Bank Founda- ba
[3] Majalah-OJK edisi Desember 2014 th 2, hal. 12: “Pelaku industri dua hal tersebut untuk meningkatkan pada holding company-nya. Jadi,